## 嵌入向量

在我們之前的範例中，我們操作的是長度為 `vocab_size` 的高維詞袋向量，並且我們明確地從低維的位置表示向量轉換為稀疏的一熱編碼表示。這種一熱編碼表示在記憶體使用上並不高效，而且每個詞彙是彼此獨立處理的，也就是說，一熱編碼向量並不表達詞彙間的任何語義相似性。

在本單元中，我們將繼續探索 **News AG** 數據集。首先，讓我們載入資料並從先前的筆記本中取得一些定義。


In [1]:
import torch
import torchtext
import numpy as np
from torchnlp import *
train_dataset, test_dataset, classes, vocab = load_dataset()
vocab_size = len(vocab)
print("Vocab size = ",vocab_size)

Loading dataset...


d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\train.csv: 29.5MB [00:01, 18.8MB/s]                            
d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\test.csv: 1.86MB [00:00, 11.2MB/s]                          


Building vocab...
Vocab size =  95812



## 什麼是 embedding？

**embedding** 的想法是用較低維度的密集向量來表示詞彙，這些向量在某種程度上反映了詞語的語意。我們將會稍後討論如何建立有意義的詞向量，但現在先把 embedding 視為一種降低詞向量維度的方法。

因此，embedding 層會以詞彙作為輸入，並輸出指定 `embedding_size` 維度的向量。從某種意義上來說，它非常類似 `Linear` 層，但不是以 one-hot 編碼向量作為輸入，而是以詞彙的編號作為輸入。

把 embedding 層作為網路的第一層，我們就能從詞袋模型切換到 **embedding bag** 模型，在該模型中，我們先將文本中每個詞彙轉換成對應的 embedding，然後對所有這些 embedding 計算某種聚合函數，如 `sum`、`average` 或 `max`。

![展示一個五詞序列的 embedding 分類器圖片。](../../../../../translated_images/zh-TW/embedding-classifier-example.b77f021a7ee67eee.webp)

我們的分類神經網路將從 embedding 層開始，接著是聚合層，最後是線性分類器：


In [2]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, x):
        x = self.embedding(x)
        x = torch.mean(x,dim=1)
        return self.fc(x)

### 處理變長序列大小

由於此架構，我們需要以特定方式建立傳入網路的小批次。在前一單元中使用詞袋模型時，小批次中的所有 BoW 張量尺寸都相同 `vocab_size`，無論我們文字序列的實際長度為何。一旦我們改用詞嵌入，每個文字樣本的詞數就會變動，在將這些樣本組合成小批次時，我們必須使用填充（padding）。

這可以利用提供給資料來源的 `collate_fn` 函數來完成：


In [3]:
def padify(b):
    # b is the list of tuples of length batch_size
    #   - first element of a tuple = label, 
    #   - second = feature (text sequence)
    # build vectorized sequence
    v = [encode(x[1]) for x in b]
    # first, compute max length of a sequence in this minibatch
    l = max(map(len,v))
    return ( # tuple of two tensors - labels and features
        torch.LongTensor([t[0]-1 for t in b]),
        torch.stack([torch.nn.functional.pad(torch.tensor(t),(0,l-len(t)),mode='constant',value=0) for t in v])
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=padify, shuffle=True)

### 訓練嵌入分類器

現在我們已經定義了合適的資料加載器，我們可以使用先前單元中定義的訓練函式來訓練模型：


In [4]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)
train_epoch(net,train_loader, lr=1, epoch_size=25000)

3200: acc=0.6415625
6400: acc=0.6865625
9600: acc=0.7103125
12800: acc=0.726953125
16000: acc=0.739375
19200: acc=0.75046875
22400: acc=0.7572321428571429


(0.889799795315499, 0.7623160588611644)

> <strong>注意</strong>：這裡我們只訓練 25k 筆資料（不到一個完整的 epoch）以節省時間，但你可以繼續訓練，撰寫一個函式來訓練多個 epoch，並嘗試學習率參數以達到更高的準確率。你應該能夠達到約 90% 的準確率。


### EmbeddingBag 層與變長序列表示法

在先前的架構中，我們需要將所有序列填充到相同長度，以便將它們放入一個 minibatch 中。這並不是表示變長序列的最高效方式——另一種方法是使用 **offset** 向量，它會保存所有序列在一個大型向量中的偏移量。

![顯示偏移序列表示的圖片](../../../../../translated_images/zh-TW/offset-sequence-representation.eb73fcefb29b46ee.webp)

> <strong>注意</strong>：在上方圖片中，我們展示的是字元序列，但在我們的範例中，我們處理的是詞序列。然而，使用 offset 向量表示序列的一般原理仍然相同。

為了使用 offset 表示法，我們使用 [`EmbeddingBag`](https://pytorch.org/docs/stable/generated/torch.nn.EmbeddingBag.html) 層。它類似於 `Embedding`，但它以內容向量和偏移向量作為輸入，且它還包含平均層，該層可以是 `mean`、`sum` 或 `max`。

以下是使用 `EmbeddingBag` 的修改後網路：


In [5]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.EmbeddingBag(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, text, off):
        x = self.embedding(text, off)
        return self.fc(x)

為了準備訓練用的資料集，我們需要提供一個轉換函式來準備偏移向量：


In [6]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1])) for t in b]
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)

注意，與之前所有的範例不同，我們的網路現在接受兩個參數：資料向量和偏移向量，這兩者大小不同。類似地，我們的資料載入器也提供了 3 個值而非 2 個：文字向量和偏移向量都作為特徵提供。因此，我們需要稍微調整訓練函數來處理這個問題：


In [7]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)

def train_epoch_emb(net,dataloader,lr=0.01,optimizer=None,loss_fn = torch.nn.CrossEntropyLoss(),epoch_size=None, report_freq=200):
    optimizer = optimizer or torch.optim.Adam(net.parameters(),lr=lr)
    loss_fn = loss_fn.to(device)
    net.train()
    total_loss,acc,count,i = 0,0,0,0
    for labels,text,off in dataloader:
        optimizer.zero_grad()
        labels,text,off = labels.to(device), text.to(device), off.to(device)
        out = net(text, off)
        loss = loss_fn(out,labels) #cross_entropy(out,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss
        _,predicted = torch.max(out,1)
        acc+=(predicted==labels).sum()
        count+=len(labels)
        i+=1
        if i%report_freq==0:
            print(f"{count}: acc={acc.item()/count}")
        if epoch_size and count>epoch_size:
            break
    return total_loss.item()/count, acc.item()/count


train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6153125
6400: acc=0.6615625
9600: acc=0.6932291666666667
12800: acc=0.715078125
16000: acc=0.7270625
19200: acc=0.7382291666666667
22400: acc=0.7486160714285715


(22.771553103007037, 0.7551983365323096)


## 語意嵌入：Word2Vec

在我們之前的範例中，模型的嵌入層學會將詞語映射到向量表示，但這種表示沒有太多語意上的意義。如果能學到這樣的向量表示，使得相似詞或同義詞在某種向量距離（例如歐氏距離）上彼此接近，將會非常有幫助。

為此，我們需要以特定方式在大規模文本集合上預先訓練我們的嵌入模型。訓練語意嵌入的最早方法之一稱為 [Word2Vec](https://en.wikipedia.org/wiki/Word2vec)。它基於兩種主要架構，用來產生詞語的分散式表示：

 - **連續詞袋模型（Continuous bag-of-words，CBoW）** — 在這種架構中，我們訓練模型從上下文環境預測一個詞。給定n-gram $(W_{-2},W_{-1},W_0,W_1,W_2)$，模型的目標是從 $(W_{-2},W_{-1},W_1,W_2)$ 預測 $W_0$。
 - **連續跳字模型（Continuous skip-gram）** 則與CBoW相反。該模型利用周圍的上下文詞語來預測當前詞。

CBoW較快，而跳字模型較慢，但對於表示不常見的詞效果較佳。

![圖片顯示將詞轉換為向量的CBoW與跳字算法。](../../../../../translated_images/zh-TW/example-algorithms-for-converting-words-to-vectors.fbe9207a726922f6.webp)

若要使用在Google News數據集上預訓練的word2vec嵌入進行實驗，我們可以使用 **gensim** 函式庫。以下找出與 'neural' 最接近的詞語。

> **注意：** 當你首次創建詞向量時，下載可能需要一些時間！


In [8]:
import gensim.downloader as api
w2v = api.load('word2vec-google-news-300')

In [9]:
for w,p in w2v.most_similar('neural'):
    print(f"{w} -> {p}")

neuronal -> 0.7804799675941467
neurons -> 0.7326500415802002
neural_circuits -> 0.7252851724624634
neuron -> 0.7174385190010071
cortical -> 0.6941086649894714
brain_circuitry -> 0.6923246383666992
synaptic -> 0.6699118614196777
neural_circuitry -> 0.6638563275337219
neurochemical -> 0.6555314064025879
neuronal_activity -> 0.6531826257705688


我們也可以從詞彙計算向量嵌入，以用於訓練分類模型（我們僅顯示向量的前20個分量以便清晰）： 


In [10]:
w2v.word_vec('play')[:20]

array([ 0.01226807,  0.06225586,  0.10693359,  0.05810547,  0.23828125,
        0.03686523,  0.05151367, -0.20703125,  0.01989746,  0.10058594,
       -0.03759766, -0.1015625 , -0.15820312, -0.08105469, -0.0390625 ,
       -0.05053711,  0.16015625,  0.2578125 ,  0.10058594, -0.25976562],
      dtype=float32)

關於語意嵌入的好處是，你可以操作向量編碼來改變語意。例如，我們可以要求找到一個詞，其向量表示應盡可能接近詞 *king* 和 *woman*，並且盡可能遠離詞 *man*：


In [10]:
w2v.most_similar(positive=['king','woman'],negative=['man'])[0]

('queen', 0.7118192911148071)

CBoW 和 Skip-Grams 都是「預測式」嵌入，因為它們只考慮局部上下文。Word2Vec 並未利用全局上下文。 

**FastText** 在 Word2Vec 的基礎上，學習每個單詞以及每個單詞中出現的字元 n-gram 的向量表示。這些表示的數值會在每次訓練步驟中平均合併成一個向量。雖然這增加了許多額外的預訓練計算，但使得詞嵌入能編碼子詞資訊。 

另一種方法 **GloVe** 利用共現矩陣的概念，使用神經方法將共現矩陣分解成更具表達力且非線性的詞向量。

你可以通過將嵌入更改為 FastText 和 GloVe 來試玩示例，因為 gensim 支援多種不同的詞嵌入模型。


## 在 PyTorch 中使用預訓練的嵌入向量

我們可以修改上述範例，使用語義嵌入向量（如 Word2Vec）事先填充我們嵌入層的矩陣。我們需要考慮到預訓練嵌入的詞彙表和我們的文本語料庫可能不匹配，因此我們將以隨機值初始化缺少的詞彙權重：


In [11]:
embed_size = len(w2v.get_vector('hello'))
print(f'Embedding size: {embed_size}')

net = EmbedClassifier(vocab_size,embed_size,len(classes))

print('Populating matrix, this will take some time...',end='')
found, not_found = 0,0
for i,w in enumerate(vocab.get_itos()):
    try:
        net.embedding.weight[i].data = torch.tensor(w2v.get_vector(w))
        found+=1
    except:
        net.embedding.weight[i].data = torch.normal(0.0,1.0,(embed_size,))
        not_found+=1

print(f"Done, found {found} words, {not_found} words missing")
net = net.to(device)

Embedding size: 300
Populating matrix, this will take some time...Done, found 41080 words, 54732 words missing


現在讓我們訓練模型。請注意，訓練模型所需的時間明顯比前一個範例長，這是因為嵌入層的大小較大，因此參數數量也大幅增加。另外，基於這個原因，如果我們想避免過度擬合，可能需要在更多範例上訓練模型。


In [12]:
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6359375
6400: acc=0.68109375
9600: acc=0.7067708333333333
12800: acc=0.723671875
16000: acc=0.73625
19200: acc=0.7463541666666667
22400: acc=0.7560714285714286


(214.1013875559821, 0.7626759436980166)

在我們的案例中，我們沒有看到準確度有大幅提升，這很可能是因為詞彙表差異很大。
為了解決詞彙表差異的問題，我們可以使用以下其中一種解決方案：
* 重新訓練 word2vec 模型以符合我們的詞彙表
* 使用預訓練的 word2vec 模型中的詞彙表來載入我們的資料集。載入資料集時可以指定使用的詞彙表。

後者方法似乎較為簡便，特別是因為 PyTorch `torchtext` 框架內建對嵌入的支援。例如，我們可以以下列方式實例化基於 GloVe 的詞彙表：


In [14]:
vocab = torchtext.vocab.GloVe(name='6B', dim=50)

100%|█████████▉| 399999/400000 [00:15<00:00, 25411.14it/s]


載入的詞彙表具有以下基本操作：
* `vocab.stoi` 允許我們將單詞轉換為其詞典索引
* `vocab.itos` 則相反 — 將數字轉換為單詞
* `vocab.vectors` 是嵌入向量陣列，因此要取得單詞 `s` 的嵌入，我們需要使用 `vocab.vectors[vocab.stoi[s]]`

以下是操作嵌入的示例，以展示方程式 **kind-man+woman = queen**（我必須稍微調整係數使其生效）：


In [15]:
# get the vector corresponding to kind-man+woman
qvec = vocab.vectors[vocab.stoi['king']]-vocab.vectors[vocab.stoi['man']]+1.3*vocab.vectors[vocab.stoi['woman']]
# find the index of the closest embedding vector 
d = torch.sum((vocab.vectors-qvec)**2,dim=1)
min_idx = torch.argmin(d)
# find the corresponding word
vocab.itos[min_idx]

'queen'

要使用那些嵌入向量來訓練分類器，我們首先需要使用 GloVe 詞彙對資料集進行編碼：


In [16]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1],voc=vocab)) for t in b] # pass the instance of vocab to encode function!
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

如上所見，所有向量嵌入均存儲在 `vocab.vectors` 矩陣中。這使得通過簡單複製將這些權重加載到嵌入層的權重中非常簡單： 


In [17]:
net = EmbedClassifier(len(vocab),len(vocab.vectors[0]),len(classes))
net.embedding.weight.data = vocab.vectors
net = net.to(device)

現在讓我們訓練模型，看看是否會得到更好的結果：


In [18]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6271875
6400: acc=0.68078125
9600: acc=0.7030208333333333
12800: acc=0.71984375
16000: acc=0.7346875
19200: acc=0.7455729166666667
22400: acc=0.7529464285714286


(35.53972978646833, 0.7575175943698017)

我們未看到準確度顯著提升的原因之一是，我們資料集中有些詞彙在預訓練的 GloVe 詞彙表中缺失，因此這些詞彙基本上被忽略。為了克服這個問題，我們可以在自己的資料集上訓練自己的詞嵌入。 


## 上下文嵌入

傳統的預訓練嵌入表示（如 Word2Vec）的一個主要限制是詞義消歧的問題。雖然預訓練的嵌入能捕捉詞在上下文中的部分意義，但詞的所有可能意義都編碼在同一個嵌入中。這可能會在下游模型中產生問題，因為許多詞語（例如 “play”）在不同上下文中有不同的意思。

例如，詞 “play” 在以下這兩個不同的句子中意義迥異：
- 我去劇院看了一場 **play**。
- John 想和他的朋友們一起 **play**。

上述預訓練嵌入將詞 “play” 的這兩種含義都表示在相同的嵌入中。為了解決這個限制，我們需要基於 <strong>語言模型</strong> 構建嵌入，該模型是在大量語料上訓練的，並且<em>了解</em>如何在不同上下文中組合詞語。本教程不包含上下文嵌入的詳細討論，但在下一單元談論語言模型時，我們將回頭介紹它們。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
此文件已使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們努力追求準確性，但請注意自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應視為權威來源。對於關鍵資訊，建議採用專業人工翻譯。我們不對因使用此翻譯所產生的任何誤解或誤譯承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
